In [0]:
%run ./../config/00_project_config

In [0]:
%run ./../setup/00_storage_configuration

In [0]:
inventory_bronze_path = f"{BRONZE_PATH}/inventory"
inventory_silver_path = f"{SILVER_PATH}/inventory"

In [0]:
df_inventory_bronze = spark.read.format("delta") \
    .load(inventory_bronze_path)

In [0]:
df_inventory_bronze.printSchema()

In [0]:
%skip
display(df_inventory_bronze.limit(10))

In [0]:
df_inventory_silver = df_inventory_bronze

In [0]:
%skip
df_inventory_silver.write.format("delta") \
    .mode("append") \
    .save(inventory_silver_path)

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forPath(
    spark,
    inventory_silver_path
)

silver_table.alias("target") \
.merge(
    df_inventory_silver.alias("source"),
    "target.inventory_id = source.inventory_id"
) \
.whenMatchedUpdate(
    set = {
        "warehouse_id": "source.warehouse_id",
        "product_id": "source.product_id",
        "quantity_available": "source.quantity_available",
        "quantity_reserved": "source.quantity_reserved",
        "last_updated": "source.last_updated"
    }
) \
.whenNotMatchedInsertAll() \
.execute()

In [0]:
spark.read.format("delta") \
    .load(inventory_silver_path) \
    .count()